# Primitive Design

In [1]:
import numpy as np
import sysl.symbolic as sls
import geolipi.symbolic as gls
from sysl.shader.evaluate import evaluate_to_shader
from sysl.shader_runtime.generate_shader_html import create_shader_html, make_jupyter_compatible_html
from sysl.shader_runtime.generate_shader_html import create_multibuffer_shader_html
from IPython.display import display, HTML
import superfit.symbolic as sps
import superfit.shader
import os
import superfit.torch_compute
SAVE_DIR = "web"
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
settings = {
    "render_mode": "v4",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": False,
        "_AA": 1,
        "_RAYCAST_MAX_STEPS": 200,
    },
    "set_to_ubo": False,
    "export_params": False,
}



In [2]:
# All 3D primitives expressions
all_primitives_expressions = [
    sps.SuperFrustum(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilation"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "TaperRatio"),
        gls.UniformFloat((-2.0,), (-1.5,), (2.0,), "Bulge"),
        gls.UniformFloat((0.0,), (0.0,), (1.0,), "OnionRatio"),
    ),
    sps.SPProto(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformVec4((0.0,0.0, 0.0, 0.0), (0.9,0.9, 0.1, 0.1), (1.0, 1.0, 1.0, 1.0), "roundness"),
        gls.UniformFloat((-0.1,), (0.01,), (1.0,), "dilation"),
        gls.UniformFloat((0.0,), (0.1,), (1.0,), "OnionRatio"),
        gls.UniformVec2((0.0, 0.0), (0.0, 1.0), (1.0, 1.0), "extrussion"),
    ),
    sps.SuperGeon(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.5,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "taper"),
        gls.UniformFloat((-2.0,), (0.0,), (2.0,), "bulge"),
        gls.UniformFloat((0.0,), (0.0,), (1.0,), "onion_ratio"),
        gls.UniformFloat((-1.0,), (0.0,), (1.0,), "trapeze"),
        gls.UniformFloat((-1.0,), (0.0,), (1.0,), "taper_bulge"),
        gls.UniformFloat((-2 * np.pi,), (0.0,), (2 * np.pi,), "rot_2d"),
    ),
    # OLD PRIMITIVES
    sps.SPTaperedOnion(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "scale"),
        gls.UniformFloat((0.0,), (0.0,), (1.0,), "onion_ratio"),
    ),
    sps.SPTaperedWrongV1(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "scale"),
    ),
    sps.SPTaperedWrongV2(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformVec2((0.0, 0.0), (0.0, 1.0), (1.0, 1.0), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformVec2((0.0, 0.0), (0.0, 1.0), (1.0, 1.0), "scale_opp"),
    ),
    sps.SPTaperedNewtonV1(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "scale"),
    ),
    sps.SPTaperedApproxV1(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "scale_opp"),
    ),
    sps.SPTaperedApproxV2(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformVec2((0.0, 0.0), (0.0, 1.0), (1.0, 1.0), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformVec2((0.0, 0.0), (0.0, 1.0), (2.0, 2.0), "scale_opp"),
    ),
    sps.SPChamferedV1(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "ch"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
    ),
    sps.SPChamferedV2(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformVec2((0.0, 0.0), (0.2, 0.2), (1.0, 1.0), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilate_3d"),
        gls.UniformVec2((0.0, 0.0), (0.0, 1.0), (2.0, 2.0), "scale_opp"),
    ),
]
var_axis_primitives = [
    sps.SuperFrustumX(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilation"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "TaperRatio"),
        gls.UniformFloat((-2.0,), (0.0,), (2.0,), "Bulge"),
        gls.UniformFloat((0.0,), (0.0,), (1.0,), "OnionRatio"),
    ),
    sps.SuperFrustumY(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilation"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "TaperRatio"),
        gls.UniformFloat((-2.0,), (0.0,), (2.0,), "Bulge"),
        gls.UniformFloat((0.0,), (0.0,), (1.0,), "OnionRatio"),
    ),
    sps.SuperFrustumZ(
        gls.UniformVec3((0.0, 0.0, 0.0,), (0.2, 0.3, 0.5), (2.0, 2.0, 2.0), "size"),
        gls.UniformFloat((0.0,), (0.75,), (1.0,), "roundness"),
        gls.UniformFloat((-0.1,), (0.0,), (1.0,), "dilation"),
        gls.UniformFloat((0.0,), (1.0,), (2.0,), "TaperRatio"),
        gls.UniformFloat((-2.0,), (0.0,), (2.0,), "Bulge"),
        gls.UniformFloat((0.0,), (0.0,), (1.0,), "OnionRatio"),
    ),
]
explicit_primitives_expressions = [
    sps.SuperGeon(
        (0.3, 0.3, 0.75),
        (0.0,),
        (0.01,), 
        (0.7,),
        (0.2,),
        (0.99,),
        (0.8,),
        (-0.99,),
        (-0.63,),
    ),
    sps.SuperGeon((0.163, 0.159, 0.055), (0.900), (0.000), (1.900), (0.000), (0.000), (0.0), (0.35), (0.000)),

    sps.SGSP(
        (0.3, 0.3, 0.75),
        (0.0, 0.01,0.7, 0.2,),
        (0.99,0.8, -0.99, -0.63,),
    ),
]


In [ ]:
# Modify this as needed.
SEL_INDEX = 0

# TO BE FIXED 16 26 30
# Remove 32
cur_expr = all_primitives_expressions[SEL_INDEX]
# cur_expr = var_axis_primitives[SEL_INDEX]
# cur_expr = explicit_primitives_expressions[SEL_INDEX]
print(cur_expr)
# cur_expr = gls.Translate3D(
#     gls.AxisAngleRotate3D(cur_expr, 
#         gls.UniformVec3((-2.0,-2.0, -2.0,), (0.0, 0.0, 0.00001), (2.0, 2.0, 2.0), "rotation"),
#     ), gls.UniformVec3((-1.0, -1.0, -1.0,), (0.0, 0.0, 0.00001), (1.0, 1.0, 1.0), "translation")
# )
# print(cur_expr)
material = sls.MatRefV4("MatWood")
# material = sls.MaterialV1((2.0,))
scene_with_material = sls.MatSolid(cur_expr, material)
# Get Shader Code
shader_info = evaluate_to_shader(scene_with_material, settings=settings, 
                                #  mode="singlepass", 
                                 mode="multipass", 
                                 post_process_shader=["part_outline_nobg"]
                                 )
# TO visualize in a browser:
html_code = create_multibuffer_shader_html(shader_info, show_controls=True, layout_horizontal=True)
# html_code = create_shader_html(*shader_info, show_controls=True, layout_horizontal=True)
# To visualize inline in jupyter notebook:
with open(os.path.join(SAVE_DIR, "test.html"), "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
display(HTML(jupy_wrapper_html))




In [ ]:
# Also check match between eval and render. 
from sysl.shader.utils.texture import recursive_encode_texture_tensor
from geolipi.torch_compute import Sketcher, recursive_evaluate
sketcher = Sketcher(n_dims=3, resolution=64)

# cur_expr = gls.ArbitraryCappedCylinder3D((0.3, 0.0, 0.1, ), (0.8, 0.0, 0.7,), (0.1,))
eval_out = recursive_evaluate(cur_expr.tensor(), sketcher)# [..., 0]
if len(eval_out.shape) == 2:
    eval_out = eval_out[..., 0]
expr = gls.SDFGrid3D(eval_out, "torch_eval", (0.2,))

expr = recursive_encode_texture_tensor(expr, sketcher)

material = sls.MatRefV4("MatPlastic")

new_expr = sls.MatSolidV1(expr, material)
scene_with_material = sls.MatSolid(cur_expr, material)

final_scene = gls.Union(
    new_expr,
    gls.Translate3D(scene_with_material, (1.0, 0.0, 0.0))
)


shader_info = evaluate_to_shader(final_scene, settings=settings,
mode="multipass")
# TO visualize in a browser:
html_code = create_multibuffer_shader_html(shader_info, show_controls=True, layout_horizontal=True)
# html_code = create_shader_html(*shader_info, show_controls=True, layout_horizontal=True)
# To visualize inline in jupyter notebook:
with open(os.path.join(SAVE_DIR, "test.html"), "w") as f:
    f.write(html_code)
# jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))
